# 06 · Shuffle, Wide vs. Narrow, Broadcast Join (Caso B)

**Teoria**: docs/03-transformacoes-acoes-dag.md, docs/06-persistencia-e-otimizacao.md

**Pré-requisito**: `make up-cluster` ainda em execução.

🎯 **Objetivo**: entender na prática a diferença entre transformações **narrow** (sem shuffle) e **wide** (com shuffle), e como o broadcast join elimina o shuffle do lado grande.

Este laboratório coloca um join sem Shuffle (`empresas`, 50 linhas → `broadcast()`) lado a lado com um com muito Shuffle (`funcionarios`, milhares de linhas → SortMergeJoin), para que você veja a diferença de custo na Spark UI, não apenas na teoria.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import sum as spark_sum

# Cria sessão Spark Connect — conecta ao servidor gRPC no container spark-connect
# (make up-cluster deve estar rodando)
spark = get_connect_session("06-shuffle-broadcast")

# Lê os dados compartilhados — caminhos /data/... resolvidos dentro dos containers
vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
empresas = spark.read.parquet(layer_path("connect", "bronze", "empresas"))
funcionarios = spark.read.parquet(layer_path("connect", "bronze", "funcionarios"))

## Broadcast join — sem Shuffle de `vendas`

📌 Procure por `BroadcastHashJoin` no plano abaixo. Depois verifique a aba Stages da Spark UI (http://localhost:4040) para este Job: nenhum nó `Exchange` para o lado grande.

🧠 **Como funciona**: o Spark copia a tabela `empresas` (50 linhas) para a memória de cada executor. O join acontece localmente, sem movimento de dados do lado `vendas`.

In [ ]:
# Broadcast join: dica explícita broadcast() — força BroadcastHashJoin
# empresas é pequena (~50 linhas) e será copiada para cada executor
broadcast_join = vendas.join(broadcast(empresas), "id_empresa")

# explain() mostra o plano físico — procure por BroadcastHashJoin
broadcast_join.explain()

# Executa o join com groupBy para materializar o resultado
# Obs.: o count() no final força a execução, mas descartamos o valor
broadcast_join.groupBy("setor").agg(spark_sum("valor")).count()

📌 **Resultado do broadcast join**:

O plano deve mostrar `BroadcastHashJoin` sem `Exchange` no lado `vendas`. Isso significa ZERO shuffle de dados — apenas uma cópia pequena de `empresas` para cada worker.

💡 **Dica**: na aba **Stages** da Spark UI, compare o número de Stages e Tasks deste Job com o próximo (shuffle join). O broadcast join tipicamente tem menos Stages.

## Shuffle join — ambos os lados são redistribuídos

📌 Procure por `SortMergeJoin` e (geralmente) dois nós `Exchange` — um para cada lado do join. Na Spark UI, compare a duração deste Stage contra o broadcast join acima.

🧠 **Como funciona**: o Spark reparticiona AMBOS os DataFrames por `id_funcionario`. Isso exige shuffle dos dois lados — muito mais caro em termos de rede e disco.

In [ ]:
# Shuffle join: sem broadcast — Catalyst escolhe SortMergeJoin
# Funcionarios tem milhares de linhas, não cabe no limite de broadcast
shuffle_join = vendas.join(funcionarios, "id_funcionario")

# explain() mostra os nós Exchange — um para vendas, outro para funcionarios
shuffle_join.explain()

# Executa com groupBy para materializar
shuffle_join.groupBy("cargo").agg(spark_sum("valor")).count()

📌 **Resultado do shuffle join**:

O plano mostra `SortMergeJoin` com dois `Exchange` (um para cada lado). Cada `Exchange` representa um shuffle — dados sendo reescritos em disco e transferidos pela rede entre executores.

⚠️ **Atenção**: o shuffle join pode ser 10×-100× mais lento que o broadcast join dependendo do volume de dados. Na Spark UI, veja a diferença na métrica **Shuffle Read/Write** (MB transferidos).

## Reparticionando uma vez, reutilizando entre operações

🧠 **Estratégia**: se você sabe que executará várias consultas `groupBy("id_empresa")` em sequência, repartitionar por essa chave antecipadamente evita pagar o custo do Shuffle mais de uma vez.

📌 O `repartition(8, "id_empresa")` faz um shuffle único e caro, mas os groupBy subsequentes aproveitam a partição já alinhada — sem novo shuffle.

In [ ]:
# Reparticiona por id_empresa em 8 partições — shuffle único e proposital
# Após isso, todos os dados com o mesmo id_empresa estão na MESMA partição
vendas_by_empresa = vendas.repartition(8, "id_empresa")
vendas_by_empresa.cache()                         # Cacheia para reuso
vendas_by_empresa.count()                         # Materializa o cache

# Ambos os groupBy reusam o mesmo particionamento — sem shuffle extra
vendas_by_empresa.groupBy("id_empresa").agg(spark_sum("valor")).show()
vendas_by_empresa.groupBy("id_empresa").count().show()

# Libera o cache — importante para não reter memória desnecessária
vendas_by_empresa.unpersist()

📌 **Por que isso funciona?**:

Quando você reparticiona por `id_empresa`, o Spark garante que todas as linhas com o mesmo `id_empresa` fiquem na mesma partição. Um `groupBy("id_empresa")` subsequente consegue agregar localmente, sem precisar de outro shuffle.

💡 **Dica técnica**: isso se chama **partition pruning** combinado com **cache** — uma técnica poderosa para pipelines que fazem múltiplas agregações pela mesma chave.

⚠️ **Atenção**: o `repartition()` em si é uma operação **wide** (shuffle). O benefício só aparece se você fizer pelo menos 2 operações após ele.

In [ ]:
# Encerra a sessão Spark Connect — libera recursos no cluster
spark.stop()